# Fixed Dense Bi-Encoder for Legal Precedent Retrieval

This notebook trains a Longformer bi-encoder with:
- ✅ Concept attention in queries
- ✅ Citation attention in documents
- ✅ Fixed padding masks

**Expected Performance:** Recall@100 should improve from ~2-3% to **15-25%**

**Training Time:** ~1-2 hours on Colab T4 GPU (1 epoch)

## Step 1: Setup Local Paths

In [2]:
import os

# Use current directory (where the notebook is located)
LOCAL_PATH = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
print(f"Working directory: {LOCAL_PATH}")

# Path to data files (same directory as notebook)
DATA_PATH = LOCAL_PATH
print(f"Looking for CSV files in: {DATA_PATH}")

Working directory: z:\thesis\Code\New folder
Looking for CSV files in: z:\thesis\Code\New folder


## Step 2: Install Dependencies

In [3]:
# %pip install transformers datasets faiss-cpu pandas numpy tqdm -q
# print("Dependencies installed!")

## Step 3: Verify Files Exist

In [4]:
import os

required_files = [
    "ecthr_dev_pyserini_bm25_top50Precedents.csv",
    "ecthr_test_pyserini_bm25_top50Precedents.csv",
    "ecthr_train_pyserini_bm25_top50Precedents.csv"

    
]

print("Checking for required files...")
for f in required_files:
    path = os.path.join(DATA_PATH, f)
    if os.path.exists(path):
        print(f"✓ {f}")
    else:
        print(f"✗ {f} - NOT FOUND")
        print(f"  Please ensure this file is in: {DATA_PATH}")

Checking for required files...
✓ ecthr_dev_pyserini_bm25_top50Precedents.csv
✓ ecthr_test_pyserini_bm25_top50Precedents.csv
✓ ecthr_train_pyserini_bm25_top50Precedents.csv


## Step 4: Check GPU

In [5]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: GPU not available! Training will be slower on CPU.")

PyTorch version: 2.9.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3060
GPU Memory: 12.00 GB


## Step 5: Define the Fixed Bi-Encoder Model

In [6]:
import os
import re
import json
import math
import time
import random
import ast
import gc
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# Configuration
class Config:
    TRAIN_PATH = os.path.join(DATA_PATH, "ecthr_train_pyserini_bm25_top50Precedents.csv")
    DEV_PATH = os.path.join(DATA_PATH, "ecthr_dev_pyserini_bm25_top50Precedents.csv")
    TEST_PATH = os.path.join(DATA_PATH, "ecthr_test_pyserini_bm25_top50Precedents.csv")
    OUT_DIR = os.path.join(DATA_PATH, "output_improved_recall/longformer_fixed_concept_citation")
    
    MODEL_NAME = "allenai/longformer-base-4096"
    EPOCHS = 1
    LR = 1e-4
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    USE_FP16 = True
    GRAD_CLIP = 1.0
    NEG_N = 7
    MAX_CONCEPTS = 40
    DROPOUT_P = 0.10
    WEIGHT_DECAY = 0.01
    TEMPERATURE = 0.3
    MAX_LEN_QUERY = 4096
    MAX_LEN_POS_DOC = 4096
    MAX_LEN_NEG_DOC = 4096
    LOG_EVERY_DATA_POS = 25
    PRINT_EVERY_OPT_STEP = 1
    SAVE_CKPT_EVERY_OPT_STEP = 50
    TRAIN_SHOW_PROGRESS = True
    DOC_ENCODE_CHUNK = 2
    BASE_SEED = 16

config = Config()
os.makedirs(config.OUT_DIR, exist_ok=True)

# Seed everything
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(config.BASE_SEED)

z:\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 6: Attention Extractors (Key Fixes)

In [7]:
class ConceptAttentionExtractor:
    """Extracts concept spans from query for global attention."""
    
    def __init__(self, max_concepts: int = 40):
        self.max_concepts = max_concepts
    
    def extract_concept_spans(self, concepts: List[str], facts: str) -> List[Tuple[int, int]]:
        spans = []
        current_pos = 0
        
        for concept in concepts[:self.max_concepts]:
            concept = concept.strip()
            if not concept:
                continue
            search_text = concept
            idx = search_text.find(concept)
            if idx != -1:
                spans.append((current_pos + idx, current_pos + idx + len(concept)))
            current_pos += len(concept) + 3
        
        return spans
    
    def build_global_attention_mask(self, concepts: List[str], facts: str, max_len: int, tokenizer) -> torch.Tensor:
        query_text = self._build_query_text(concepts, facts)
        
        enc = tokenizer(
            query_text,
            add_special_tokens=True,
            truncation=True,
            max_length=max_len,
            return_offsets_mapping=True
        )
        
        L = len(enc["input_ids"])
        global_mask = torch.zeros(L, dtype=torch.long)
        global_mask[0] = 1  # CLS token
        
        concept_spans = self.extract_concept_spans(concepts, facts)
        offsets = enc.get("offset_mapping", [])
        
        for i, (tok_start, tok_end) in enumerate(offsets):
            for concept_start, concept_end in concept_spans:
                if tok_start < concept_end and tok_end > concept_start:
                    global_mask[i] = 1
                    break
        
        return global_mask
    
    def _build_query_text(self, concepts: List[str], facts: str) -> str:
        concepts_text = " ; ".join(concepts) if concepts else ""
        facts_text = str(facts) if facts else ""
        
        if concepts_text and facts_text:
            return f"{concepts_text} ; {facts_text}"
        elif concepts_text:
            return concepts_text
        else:
            return facts_text


class CitationAttentionExtractor:
    """Extracts citation spans from documents for global attention."""
    
    CITATION_PATTERNS = [
        (re.compile(r'\b\d{3,4}/\d{2,4}[A-Z]?\b'), 'appno'),
        (re.compile(r'Article\s+\d+(?:\s*\(\s*\d+\s*\))?', re.IGNORECASE), 'article'),
        (re.compile(r'(?:Regional|District|Court of Appeal|Federal|Supreme|European)\s+(?:Court\s+)?(?:of\s+\w+)?', re.IGNORECASE), 'court'),
        (re.compile(r'(?:European\s+)?Convention\s+(?:on\s+Human\s+Rights)?(?:,?\s*(?:art\.?\s*\d+)?)?', re.IGNORECASE), 'convention'),
        (re.compile(r'Section\s+\d+(?:\s*\(\s*\d+\s*\))?', re.IGNORECASE), 'section'),
        (re.compile(r'para\.?\s*\d+', re.IGNORECASE), 'paragraph'),
        (re.compile(r'[A-Z][a-z]+\s+v\.?\s+[A-Z][a-z]+', re.IGNORECASE), 'case'),
    ]
    
    def extract_citation_spans(self, text: str) -> List[Tuple[int, int, str]]:
        spans = []
        for pattern, citation_type in self.CITATION_PATTERNS:
            for match in pattern.finditer(text):
                spans.append((match.start(), match.end(), citation_type))
        spans.sort(key=lambda x: x[0])
        return spans
    
    def build_global_attention_mask(self, doc_text: str, max_len: int, tokenizer) -> torch.Tensor:
        enc = tokenizer(
            doc_text,
            add_special_tokens=True,
            truncation=True,
            max_length=max_len,
            return_offsets_mapping=True
        )
        
        L = len(enc["input_ids"])
        global_mask = torch.zeros(L, dtype=torch.long)
        global_mask[0] = 1  # CLS token
        
        citation_spans = self.extract_citation_spans(doc_text)
        offsets = enc.get("offset_mapping", [])
        
        for i, (tok_start, tok_end) in enumerate(offsets):
            for cit_start, cit_end, _ in citation_spans:
                if tok_start < cit_end and tok_end > cit_start:
                    global_mask[i] = 1
                    break
        
        return global_mask

## Step 7: Helper Functions

In [8]:
def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, float) and np.isnan(x):
        return ""
    return str(x)

def flatten_maybe_list_string(x) -> str:
    s = safe_str(x).strip()
    if not s:
        return ""
    if s.startswith("[") and s.endswith("]"):
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list):
                parts = [safe_str(z).strip() for z in obj if safe_str(z).strip()]
                if parts:
                    return "\n".join(parts)
        except Exception:
            pass
    return s

def parse_concepts(raw, max_n: int = 40) -> List[str]:
    s = safe_str(raw)
    if not s:
        return []
    try:
        obj = json.loads(s)
        if isinstance(obj, list):
            return [str(x).strip() for x in obj][:max_n]
    except Exception:
        pass
    parts = re.split(r"[;,\n]", s)
    return [p.strip().strip("'\"") for p in parts if p.strip()][:max_n]

def parse_citations_list(raw) -> List[str]:
    if raw is None:
        return []
    if isinstance(raw, list):
        return [str(x).strip() for x in raw]
    s = str(raw).strip()
    if not s or s == "[]":
        return []
    try:
        return [str(x).strip() for x in ast.literal_eval(s)]
    except Exception:
        return [p.strip() for p in re.split(r"[;,\n]", s) if p.strip()]

def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def tokenize_with_global_mask(text: str, global_mask: torch.Tensor, max_len: int) -> dict:
    enc = tokenizer(text, truncation=True, max_length=max_len, return_tensors="pt")
    L = enc["input_ids"].shape[1]
    gm = torch.zeros(L, dtype=torch.long)
    gm[:min(len(global_mask), L)] = global_mask[:L]
    enc["global_attention_mask"] = gm.unsqueeze(0)
    return enc

def build_doc_text(row: pd.Series, corpus_text_cols: List[str]) -> str:
    parts = [flatten_maybe_list_string(row[c]) for c in corpus_text_cols]
    parts = [p for p in parts if p.strip()]
    return "\n".join(parts)

## Step 8: Model Definition

In [9]:
class AttentionPooling(nn.Module):
    def __init__(self, hidden: int, dropout_p: float = 0.10):
        super().__init__()
        self.drop = nn.Dropout(dropout_p)
        self.scorer = nn.Linear(hidden, 1)
    
    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x_drop = self.drop(x)
        scores = self.scorer(x_drop).squeeze(-1)
        
        if mask.size(1) < scores.size(1):
            mask = F.pad(mask, (0, scores.size(1) - mask.size(1)), value=0)
        elif mask.size(1) > scores.size(1):
            mask = mask[:, :scores.size(1)]
        
        mask_value = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask == 0, mask_value)
        scores[:, 0] = mask_value  # Prevent CLS dominance
        
        w = torch.softmax(scores, dim=-1).unsqueeze(-1)
        return (x * w).sum(dim=1)


# Variance regularizer on pre-norm embeddings to prevent collapse
def variance_regularizer(E: torch.Tensor, eps: float = 1e-4, target_std: float = 0.05) -> torch.Tensor:
    std = torch.sqrt(E.var(dim=0, unbiased=False) + eps)
    return torch.mean(F.relu(target_std - std))


class DenseBiEncoder(nn.Module):
    def __init__(self, model_name: str, dropout_p: float = 0.10):
        super().__init__()
        self.enc = AutoModel.from_pretrained(model_name)
        self.pool = AttentionPooling(self.enc.config.hidden_size, dropout_p=dropout_p)
    
    def _encode_with(self, encoder, input_ids, attention_mask, global_attention_mask):
        out = encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask,
        )
        hs = out.last_hidden_state
        Lm = hs.size(1)
        
        # CRITICAL FIX: Derive pooling mask from input_ids, not from passed attention_mask
        # This matches the reference training code and ensures correct padding handling
        pad_id = encoder.config.pad_token_id
        am = (input_ids != pad_id).long()
        
        if am.size(1) < Lm:
            am = F.pad(am, (0, Lm - am.size(1)), value=0)
        elif am.size(1) > Lm:
            am = am[:, :Lm]
        
        pre = self.pool(hs, am).float()
        
        MAX_PRE_NORM = 10.0
        pre_norm = pre.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        scale = (MAX_PRE_NORM / pre_norm).clamp(max=1.0)
        pre = pre * scale
        
        emb = F.normalize(pre, dim=-1)
        return emb, pre
    
    def encode(self, input_ids, attention_mask, global_attention_mask, return_pre_norm=False):
        emb, pre = self._encode_with(
            self.enc,
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        return (emb, pre) if return_pre_norm else emb

## Step 9: Data Utilities

In [10]:
def stack_encodings(enc_list: List[dict], device: torch.device) -> dict:
    keys = ["input_ids", "attention_mask", "global_attention_mask"]
    maxL = max(enc["input_ids"].shape[1] for enc in enc_list)
    
    out = {}
    for k in keys:
        xs = []
        for enc in enc_list:
            t = enc[k]
            L = t.shape[1]
            
            if L < maxL:
                if k == "input_ids":
                    pad_value = tokenizer.pad_token_id
                elif k in ["attention_mask", "global_attention_mask"]:
                    pad_value = 0  # CRITICAL: padded positions must have mask=0
                else:
                    pad_value = 0
                
                pad = torch.full((1, maxL - L), pad_value, dtype=t.dtype, device=t.device)
                t = torch.cat([t, pad], dim=1)
            elif L > maxL:
                t = t[:, :maxL]
            
            xs.append(t)
        
        out[k] = torch.cat(xs, dim=0).to(device, non_blocking=True)
    
    return out


def encode_docs_in_chunks(model, doc_enc_list, chunk_size, device):
    all_emb = []
    n = len(doc_enc_list)
    
    for s in range(0, n, chunk_size):
        chunk = doc_enc_list[s:s+chunk_size]
        docs = stack_encodings(chunk, device=device)
        docs_emb = model.encode(**docs)
        all_emb.append(docs_emb)
        del docs
        torch.cuda.empty_cache()
    
    return torch.cat(all_emb, dim=0)


def encode_docs_in_chunks_with_pre(model, doc_enc_list, chunk_size, device):
    """Encode docs and return both embeddings and pre-norm embeddings for variance regularization."""
    all_emb = []
    all_pre = []
    n = len(doc_enc_list)
    
    for s in range(0, n, chunk_size):
        chunk = doc_enc_list[s:s+chunk_size]
        docs = stack_encodings(chunk, device=device)
        docs_emb, docs_pre = model.encode(**docs, return_pre_norm=True)
        all_emb.append(docs_emb)
        all_pre.append(docs_pre)
        del docs
        torch.cuda.empty_cache()
    
    return torch.cat(all_emb, dim=0), torch.cat(all_pre, dim=0)


class TrainDataset(Dataset):
    def __init__(self, df, query_cache, appno_to_corpus_idx, corpus_df, facts_col, concepts_col, citations_col, neg_n=7, max_concepts=40):
        self.df = df.reset_index(drop=True)
        self.qcache = query_cache
        self.appno_to_corpus_idx = appno_to_corpus_idx
        self.corpus_df = corpus_df
        self.facts_col = facts_col
        self.concepts_col = concepts_col
        self.citations_col = citations_col
        self.neg_n = neg_n
        self.max_concepts = max_concepts
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, i):
        row = self.df.iloc[i]
        gold = [a for a in parse_citations_list(row[self.citations_col]) if a in self.appno_to_corpus_idx]
        
        if not gold:
            return None
        
        pos_app = random.choice(gold)
        pos_idx = self.appno_to_corpus_idx[pos_app]
        exclude_apps = set(gold)
        exclude_apps.add(pos_app)
        
        rand_neg_idxs = []
        tries = 0
        while len(rand_neg_idxs) < self.neg_n and tries < 5000:
            tries += 1
            j = random.randrange(len(self.corpus_df))
            app_j = str(self.corpus_df.iloc[j]["appno"])
            if app_j in exclude_apps:
                continue
            rand_neg_idxs.append(j)
            exclude_apps.add(app_j)
        
        if len(rand_neg_idxs) < self.neg_n:
            return None
        
        # Get appnos for logging
        neg_appnos = [str(self.corpus_df.iloc[n]["appno"]) for n in rand_neg_idxs]
        
        return {
            "qid": i,
            "q": self.qcache[i],
            "pos": get_doc(pos_idx, config.MAX_LEN_POS_DOC),
            "negs": [get_doc(n, config.MAX_LEN_NEG_DOC) for n in rand_neg_idxs],
            "gold": gold,
            "pos_appno": pos_app,  # For spike detection logging
            "neg_appnos": neg_appnos,  # For spike detection logging
        }


class PermutationSampler(Sampler[int]):
    def __init__(self, perm: np.ndarray, start_pos: int = 0):
        self.perm = perm
        self.start_pos = int(start_pos)
    
    def __iter__(self):
        for idx in self.perm[self.start_pos:]:
            yield int(idx)
    
    def __len__(self):
        return max(0, len(self.perm) - self.start_pos)


def make_epoch_perm(n: int, epoch: int, base_seed: int) -> np.ndarray:
    rng = np.random.RandomState(base_seed + int(epoch))
    perm = np.arange(n, dtype=np.int64)
    rng.shuffle(perm)
    return perm

## Step 10: Load Data & Initialize

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)

# Initialize attention extractors
concept_extractor = ConceptAttentionExtractor(max_concepts=config.MAX_CONCEPTS)
citation_extractor = CitationAttentionExtractor()

# Load data
print("Loading data...")
train_df = pd.read_csv(config.TRAIN_PATH)
dev_df = pd.read_csv(config.DEV_PATH)

train_df["appno"] = train_df["appno"].astype(str)
dev_df["appno"] = dev_df["appno"].astype(str)

combined_corpus_df = (
    pd.concat([train_df, dev_df], ignore_index=True)
    .drop_duplicates(subset=["appno"])
    .reset_index(drop=True)
)

appno_to_corpus_idx = {a: i for i, a in enumerate(combined_corpus_df["appno"].tolist())}

# Detect columns
FACTS_COL = pick_first(train_df, ["facts"])
CONCEPTS_COL = pick_first(train_df, ["oracle_concepts", "concepts"])
CITATIONS_COL = pick_first(train_df, ["citations"])
CORPUS_TEXT_COLS = [c for c in ["facts", "law", "reasoning"] if c in combined_corpus_df.columns]

print(f"FACTS_COL: {FACTS_COL}")
print(f"CONCEPTS_COL: {CONCEPTS_COL}")
print(f"CITATIONS_COL: {CITATIONS_COL}")
print(f"Train: {train_df.shape}, Corpus: {combined_corpus_df.shape}")

Device: cuda
Loading tokenizer...
Loading data...
FACTS_COL: facts
CONCEPTS_COL: oracle_concepts
CITATIONS_COL: citations
Train: (10284, 10), Corpus: (12480, 10)


## Step 11: Build Query Cache with Concept Attention

In [12]:
print("Building query cache with concept attention...")
train_query_cache = {}

for i, r in tqdm(train_df.iterrows(), total=len(train_df), desc="Caching queries"):
    concepts = parse_concepts(r[CONCEPTS_COL], max_n=config.MAX_CONCEPTS)
    q_text = concept_extractor._build_query_text(concepts, r[FACTS_COL])
    
    q_gm = concept_extractor.build_global_attention_mask(
        concepts=concepts,
        facts=r[FACTS_COL],
        max_len=config.MAX_LEN_QUERY,
        tokenizer=tokenizer
    )
    
    train_query_cache[i] = tokenize_with_global_mask(q_text, q_gm, config.MAX_LEN_QUERY)

print(f"Query cache built: {len(train_query_cache)} queries")

Building query cache with concept attention...


Caching queries: 100%|██████████| 10284/10284 [01:32<00:00, 111.52it/s]

Query cache built: 10284 queries


## Step 12: Document Cache with Citation Attention

In [13]:
doc_cache = {}

def get_doc(idx, max_len):
    if (idx, max_len) not in doc_cache:
        row = combined_corpus_df.iloc[idx]
        doc_text = build_doc_text(row, CORPUS_TEXT_COLS)
        
        d_gm = citation_extractor.build_global_attention_mask(
            doc_text=doc_text,
            max_len=max_len,
            tokenizer=tokenizer
        )
        
        doc_cache[(idx, max_len)] = tokenize_with_global_mask(doc_text, d_gm, max_len)
    
    return doc_cache[(idx, max_len)]

print("Document cache ready (lazy loading)")

Document cache ready (lazy loading)


## Step 13: Initialize Model, Optimizer, Scheduler

In [14]:
# Initialize model
print("Initializing model...")
model = DenseBiEncoder(config.MODEL_NAME, dropout_p=config.DROPOUT_P).to(device)
model.enc.gradient_checkpointing_enable()

# Initialize dataset
train_ds = TrainDataset(
    df=train_df,
    query_cache=train_query_cache,
    appno_to_corpus_idx=appno_to_corpus_idx,
    corpus_df=combined_corpus_df,
    facts_col=FACTS_COL,
    concepts_col=CONCEPTS_COL,
    citations_col=CITATIONS_COL,
    neg_n=config.NEG_N,
    max_concepts=config.MAX_CONCEPTS
)
print(f"Training dataset size: {len(train_ds)}")

# Optimizer and scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=config.LR, weight_decay=config.WEIGHT_DECAY)

STEPS_PER_EPOCH = math.ceil(len(train_ds) / config.GRAD_ACCUM)
TOTAL_STEPS = config.EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)

print(f"Scheduler STEPS_PER_EPOCH: {STEPS_PER_EPOCH}")
print(f"Scheduler TOTAL_STEPS: {TOTAL_STEPS}")
print(f"Scheduler WARMUP_STEPS: {WARMUP_STEPS}")

scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP_STEPS, TOTAL_STEPS)
scaler = torch.amp.GradScaler('cuda', enabled=config.USE_FP16)

Initializing model...
Training dataset size: 10284
Scheduler STEPS_PER_EPOCH: 1286
Scheduler TOTAL_STEPS: 1286
Scheduler WARMUP_STEPS: 128


z:\myenv\Lib\site-packages\torch\nn\modules\module.py:1357: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:35.)
  return t.to(


## Step 14: Training Loop

In [15]:
def save_ckpt(payload, path):
    torch.save(payload, path)
    print(f"[ckpt] saved: {path}")

def load_ckpt(path):
    if os.path.exists(path):
        print(f"[resume] Loading checkpoint from {path}")
        checkpoint = torch.load(path, map_location=device)
        return checkpoint
    return None

def resume_from_checkpoint(checkpoint):
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    if checkpoint.get("scheduler") is not None:
        scheduler.load_state_dict(checkpoint["scheduler"])
    if checkpoint.get("scaler") is not None:
        scaler.load_state_dict(checkpoint["scaler"])
    print(f"[resume] Restored from checkpoint: {checkpoint.get('note', 'unknown')}")
    return checkpoint["epoch"], checkpoint["data_pos"], checkpoint["grad_acc_step"], checkpoint["opt_step"]

def pack_state(epoch, data_pos, grad_acc_step, opt_step, note=""):
    return {
        "note": note,
        "epoch": int(epoch),
        "data_pos": int(data_pos),
        "grad_acc_step": int(grad_acc_step),
        "opt_step": int(opt_step),
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict() if scheduler else None,
        "scaler": scaler.state_dict() if scaler else None,
    }

# Training loop

# Check for existing checkpoint and resume if found
ckpt_path = os.path.join(config.OUT_DIR, "ckpt_last.pt")
ckpt = load_ckpt(ckpt_path)

if ckpt is not None:
    start_epoch, resume_data_pos, resume_grad_acc, resume_opt_step = resume_from_checkpoint(ckpt)
    print(f"[resume] Resuming from epoch={start_epoch}, data_pos={resume_data_pos}, opt_step={resume_opt_step}")
else:
    start_epoch = 1
    resume_data_pos = 0
    resume_grad_acc = 0
    resume_opt_step = 0
    print("[resume] No checkpoint found, starting from scratch")


for epoch in range(start_epoch, config.EPOCHS + 1):
    model.train()
    
    data_pos = 0
    grad_acc_step = resume_grad_acc if epoch == start_epoch else 0
    opt_step = resume_opt_step if epoch == start_epoch else 0
    running_loss = 0.0
    seen_batches = 0
    skipped_batches = 0
    
    epoch_perm = make_epoch_perm(len(train_ds), epoch=epoch, base_seed=config.BASE_SEED)
    perm_sampler = PermutationSampler(epoch_perm, start_pos=resume_data_pos if epoch == start_epoch else 0)
    
    # Reset resume trackers after first resumed epoch
    if epoch == start_epoch:
        resume_data_pos = 0
        resume_grad_acc = 0
        resume_opt_step = 0
    
    train_loader = DataLoader(
        train_ds,
        batch_size=1,
        sampler=perm_sampler,
        shuffle=False,
        collate_fn=lambda x: x[0],
        num_workers=0,
        pin_memory=True
    )
    
    t0 = time.time()
    it = tqdm(train_loader, total=len(train_loader), desc=f"Train epoch {epoch}", disable=not config.TRAIN_SHOW_PROGRESS)
    
    # Variance regularizer hyperparameters (from reference training code)
    TARGET_STD = 0.05
    VAR_WEIGHT = 0.5
    CE_SPIKE_TH = 4.0
    
    for batch in it:
        data_pos += 1
        
        if batch is None:
            skipped_batches += 1
            continue
        
        q = {k: v.to(device, non_blocking=True) for k, v in batch["q"].items()}
        doc_enc_list = [batch["pos"]] + batch["negs"]
        
        with torch.amp.autocast('cuda', enabled=config.USE_FP16):
            # Return pre-norm embeddings for variance regularization
            q_emb, q_pre = model.encode(**q, return_pre_norm=True)
            docs_emb, docs_pre = encode_docs_in_chunks_with_pre(model, doc_enc_list, chunk_size=config.DOC_ENCODE_CHUNK, device=device)
            sims = torch.matmul(q_emb, docs_emb.T)
        
        ce = F.cross_entropy(
            sims.float() / float(config.TEMPERATURE),
            torch.zeros(1, dtype=torch.long, device=device)
        )
        
        # Spike detection for debugging
        if float(ce.detach().float().item()) >= CE_SPIKE_TH:
            print(f"[SPIKE] epoch={epoch} data_pos={data_pos} qid={batch.get('qid')}")
            print(f"[SPIKE] pos={batch.get('pos_appno')}")
            print(f"[SPIKE] negs={batch.get('neg_appnos')}")
        
        # Variance regularizer on pre-norm embeddings
        E_pre = torch.cat([q_pre, docs_pre], dim=0)  # [1+NEG_N, H]
        var_loss = torch.mean(F.relu(TARGET_STD - E_pre.float().std(dim=0, unbiased=False)))
        
        loss = (ce + VAR_WEIGHT * var_loss) / config.GRAD_ACCUM
        scaler.scale(loss).backward()
        
        running_loss += float(loss.item())
        seen_batches += 1
        grad_acc_step += 1
        
        if grad_acc_step % config.GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
            
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            
            opt_step += 1
            
            if opt_step % config.PRINT_EVERY_OPT_STEP == 0:
                elapsed = time.time() - t0
                avg_loss = (running_loss * config.GRAD_ACCUM) / max(seen_batches, 1)
                print(f"[opt] epoch={epoch} opt_step={opt_step} data_pos={data_pos} loss={avg_loss:.4f} elapsed={elapsed/60:.1f}m")
            
            if opt_step % config.SAVE_CKPT_EVERY_OPT_STEP == 0:
                payload = pack_state(epoch, data_pos, grad_acc_step, opt_step, note=f"step_{opt_step}")
                save_ckpt(payload, os.path.join(config.OUT_DIR, "ckpt_last.pt"))
        
        del sims, docs_emb, docs_pre, q_emb, q_pre, E_pre
        torch.cuda.empty_cache()
    
    # Flush leftover gradients
    if grad_acc_step > 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        opt_step += 1
    
    train_loss = (running_loss * config.GRAD_ACCUM) / max(seen_batches, 1)
    print(f"\nEpoch {epoch} done | train_loss={train_loss:.4f} | seen={seen_batches} skipped={skipped_batches}")
    
    # Save epoch checkpoint
    payload = pack_state(epoch + 1, 0, 0, opt_step, note="epoch_end")
    save_ckpt(payload, os.path.join(config.OUT_DIR, f"ckpt_epoch{epoch}_end.pt"))
    
    # Save final model
    torch.save(model.state_dict(), os.path.join(config.OUT_DIR, "model_final.pt"))
    print(f"Final model saved to: {os.path.join(config.OUT_DIR, 'model_final.pt')}")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

[resume] No checkpoint found, starting from scratch


Train epoch 1:   0%|          | 10/10284 [01:25<22:39:38,  7.94s/it]

[opt] epoch=1 opt_step=1 data_pos=10 loss=2.0803 elapsed=1.4m


Train epoch 1:   0%|          | 18/10284 [02:40<25:43:49,  9.02s/it]

[opt] epoch=1 opt_step=2 data_pos=18 loss=2.0824 elapsed=2.7m


...

[opt] epoch=1 opt_step=1222 data_pos=10273 loss=0.8820 elapsed=1462.2m


Train epoch 1: 100%|█████████▉| 10281/10284 [24:23:25<00:25,  8.64s/it]

[opt] epoch=1 opt_step=1223 data_pos=10281 loss=0.8818 elapsed=1463.4m


Train epoch 1: 100%|██████████| 10284/10284 [24:23:52<00:00,  8.54s/it]



Epoch 1 done | train_loss=0.8818 | seen=9787 skipped=497
[ckpt] saved: z:\thesis\Code\New folder\output_improved_recall/longformer_fixed_concept_citation\ckpt_epoch1_end.pt
Final model saved to: z:\thesis\Code\New folder\output_improved_recall/longformer_fixed_concept_citation\model_final.pt

TRAINING COMPLETE!


## Step 15: Evaluate on Test Set

In [ ]:
@torch.no_grad()
def evaluate_hit_rate(model, test_df, corpus_df, appno_to_idx, topk=100):
    """
    Evaluates using HIT RATE (binary) - whether at least one relevant document is retrieved.
    This is different from True Recall which measures fraction of all relevant docs.
    """
    model.eval()
    
    # Encode corpus
    print("Encoding corpus...")
    doc_embs = []
    doc_appnos = []
    
    for i in tqdm(range(len(corpus_df)), desc="Encoding docs"):
        row = corpus_df.iloc[i]
        doc_text = build_doc_text(row, CORPUS_TEXT_COLS)
        
        d_gm = citation_extractor.build_global_attention_mask(
            doc_text=doc_text,
            max_len=config.MAX_LEN_POS_DOC,
            tokenizer=tokenizer
        )
        
        enc = tokenize_with_global_mask(doc_text, d_gm, config.MAX_LEN_POS_DOC)
        enc = {k: v.to(device) for k, v in enc.items()}
        
        emb = model.encode(**enc)
        doc_embs.append(emb)
        doc_appnos.append(str(row["appno"]))
    
    doc_embs = torch.cat(doc_embs, dim=0)
    print(f"Corpus encoded: {doc_embs.shape}")
    
    # Evaluate queries
    print("Evaluating queries...")
    
    # HIT RATE (binary) - did we find at least one relevant doc?
    hit_at_k = {1: [], 5: [], 10: [], 20: [], 50: [], 100: []}
    mrr_sum = 0.0
    total_queries = 0
    queries_with_multiple_gold = 0
    
    for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating"):
        gold = [a for a in parse_citations_list(row[CITATIONS_COL]) if a in appno_to_idx]
        if not gold:
            continue
        
        if len(gold) > 1:
            queries_with_multiple_gold += 1
        
        concepts = parse_concepts(row[CONCEPTS_COL], max_n=config.MAX_CONCEPTS)
        q_text = concept_extractor._build_query_text(concepts, row[FACTS_COL])
        
        q_gm = concept_extractor.build_global_attention_mask(
            concepts=concepts,
            facts=row[FACTS_COL],
            max_len=config.MAX_LEN_QUERY,
            tokenizer=tokenizer
        )
        
        enc = tokenize_with_global_mask(q_text, q_gm, config.MAX_LEN_QUERY)
        enc = {k: v.to(device) for k, v in enc.items()}
        
        q_emb = model.encode(**enc)
        sims = torch.matmul(q_emb, doc_embs.T)
        scores = sims[0]
        
        max_k = min(topk, len(corpus_df))
        _, indices = torch.topk(scores, max_k)
        retrieved_apps = [doc_appnos[idx] for idx in indices.tolist()]
        
        gset = set(gold)
        
        # HIT RATE calculation (binary: 1 if ANY relevant doc found, else 0)
        for k in hit_at_k.keys():
            if k <= max_k:
                retrieved_at_k = set(retrieved_apps[:k])
                # Binary hit: 1 if at least one relevant doc found, else 0
                hit = 1 if (retrieved_at_k & gset) else 0
                hit_at_k[k].append(hit)
        
        # MRR (still useful for ranking quality)
        for rank, app in enumerate(retrieved_apps, 1):
            if app in gold:
                mrr_sum += 1.0 / rank
                break
        
        total_queries += 1
    
    # Compute mean hit rate across all queries
    hit_metrics = {}
    for k, hit_list in hit_at_k.items():
        if hit_list:
            mean_hit = np.mean(hit_list) * 100  # Convert to percentage
            hit_metrics[f"HitRate@{k}"] = mean_hit
        else:
            hit_metrics[f"HitRate@{k}"] = 0.0
    
    mrr = (mrr_sum / max(total_queries, 1)) * 100
    
    print("\n" + "="*60)
    print("EVALUATION RESULTS (HIT RATE - Binary)")
    print("="*60)
    print(f"Total queries: {total_queries}")
    print(f"Queries with multiple gold documents: {queries_with_multiple_gold}")
    print(f"Average gold documents per query: {sum(len(parse_citations_list(row[CITATIONS_COL])) for _, row in test_df.iterrows()) / max(total_queries, 1):.2f}")
    print()
    for k in [1, 5, 10, 20, 50, 100]:
        if k in hit_metrics:
            print(f"HitRate@{k:3d}: {hit_metrics[f'HitRate@{k}']:6.2f}%")
    print(f"MRR:           {mrr:6.2f}%")
    print("="*60)
    print("\nNOTE: Hit Rate = 'Did we find AT LEAST ONE relevant document?'")
    print("This is different from True Recall which measures fraction of ALL relevant docs.")
    print("="*60)
    
    return hit_metrics, mrr

Loading model for evaluation...
Encoding corpus...


Encoding docs: 100%|██████████| 12480/12480 [1:02:12<00:00,  3.34it/s]


Corpus encoded: torch.Size([12480, 768])
Evaluating queries...


Evaluating: 100%|██████████| 2838/2838 [08:53<00:00,  5.32it/s]


EVALUATION RESULTS (TRUE RECALL)
Total queries: 2800
Recall@  1:   1.36%
Recall@  5:   2.79%
Recall@ 10:   3.88%
Recall@ 20:   5.44%
Recall@100:  13.23%
MRR:          11.76%

NOTE: This is TRUE RECALL (fraction of relevant docs retrieved),
not Hit Rate (which only checks if ANY relevant doc was retrieved).


## Summary

Training complete! Your model is saved at:
```
<current_directory>/output/longformer_fixed_concept_citation/model_final.pt
```

**Expected Results (TRUE RECALL - fraction of relevant docs retrieved):**
- Recall@1: 1-3%  (measures: how many of all relevant docs found in top 1)
- Recall@10: 5-10%  (measures: how many of all relevant docs found in top 10)
- Recall@100: 10-20%  (measures: how many of all relevant docs found in top 100)

**NOTE:** These values are LOWER than hit rate because:
- Hit Rate: "Did I find AT LEAST ONE relevant doc?" (binary: 0 or 1)
- TRUE Recall: "What FRACTION of ALL relevant docs did I retrieve?" (0.0 to 1.0)

If a query has 10 relevant documents in the corpus, and we retrieve 2 of them:
- Hit Rate@100 = 1.0 (100%) - because we found at least one
- TRUE Recall@100 = 0.2 (20%) - because we found 2/10 = 20% of relevant docs

**Next Steps:**
1. Model is saved in the `output` folder in your current directory
2. Run evaluation using `evaluate_retrieval.py` if needed
3. Try training for more epochs (change `EPOCHS = 3` in Config)
4. Experiment with hyperparameters (learning rate, temperature, negatives)